In [10]:
from datetime import datetime, timedelta
from email.message import EmailMessage
import imaplib
import email as email_parser
import os
from pathlib import Path
import random
import re
import smtplib
import time
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

SENDER_EMAIL = os.getenv("GMAIL_USER2")
APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD2")

EXCEL_FILE_PATH = os.getenv("email")
ATTACHMENT_PATH = os.getenv("Brochure")

SUBJECT = (
    "Slipform Construction Expert for RCC Chimney & Tall Structures |"
    " Nirmanshila Construction"
)


def get_html_body(name):
    """HTML version improves trust score with modern email providers."""
    return f"""\
<html>
  <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333333;">
    <p>Dear {name},</p>

    <p>Hope this email finds you well.</p>

    <p>We are writing to introduce <strong>Nirmanshila Construction</strong> - an expert partner for your slipform constuction requirements.</p>

    <p>If you are planning new expansions, modernizations, or structural maintenance, we offer end-to-end execution capabilities across the following core areas:</p>

    <ul>
      <li><strong>Tall Structural Construction:</strong> RCC Chimney, Silos, and Elevated Overhead Water Tanks.</li>
      <li><strong>Slipform Expertise:</strong> Experienced team for slipform work.</li>
      <li><strong>Thermal & Asset Protection:</strong> Refractory Brick Lining, Industrial Painting, and Protective Coatings.</li>
      <li><strong>RCC Chimney Repair/Maintenance and General Industrial Civil Works</strong>.</li>
    </ul>

    <p><strong>Why Partner with Nirmanshila?</strong></p>
    <ul>
      <li><strong>Technical Precision:</strong> From continuous-pour slipform setups to strict structural tolerances, we ensure flawless execution.</li>
      <li><strong>Operational Integrity:</strong> We pride ourselves on maintaining strict project timelines, safety standards, and transparent budgeting.</li>
      <li><strong>Turnkey Management:</strong> We handle the engineering complexities so your team can focus on core operations.</li>
    </ul>

    <p>For your review, I have attached our <strong>Company Brochure that includes company profile & project portfolio</strong>. You can also explore our capabilities online at <a href="https://www.nirmanshilaconstruction.com">www.nirmanshilaconstruction.com</a>.</p>

    <p>We would welcome the opportunity to connect to explore how we can support your upcoming projects. Let us know a good time to talk.</p>

    <p>Looking forward to hearing from you. Thanks!</p>

    <br>
    <p><strong>Rajeev Kumar</strong> &nbsp;|&nbsp; <strong>Amit Tayal</strong><br>
    +91 7500462001 &nbsp;|&nbsp; +91 9650744299</p>

    <hr style="border: none; border-top: 1px solid #cccccc; margin: 20px 0;">
    <p style="font-size: 0.9em; color: #555555;">
      <strong>M/s Nirmanshila Construction</strong><br>
      Registered Office: 17, Kushi Vihar, Shanti Nagar, Muzaffarnagar, 251001<br>
      Website: <a href="https://www.nirmanshilaconstruction.com">www.nirmanshilaconstruction.com</a><br>
      Email: info@nirmanshilaconstruction.com
    </p>
  </body>
</html>
"""


def create_email(recipient_data, pdf_data, pdf_name):
    msg = EmailMessage()
    msg["Subject"] = SUBJECT
    msg["From"] = f"Nirmanshila Construction <{SENDER_EMAIL}>"
    msg["To"] = recipient_data["email"]

    plain_text = f"Dear {recipient_data['name']},\n\nPlease view this email in an HTML-compatible client."
    msg.set_content(plain_text)
    msg.add_alternative(get_html_body(recipient_data["name"]), subtype="html")

    if pdf_data:
        msg.add_attachment(
            pdf_data,
            maintype="application",
            subtype="pdf",
            filename=pdf_name,
        )

    return msg


def check_bounces_via_imap():
    """Connects to Gmail IMAP to search for Mail Delivery Failure bounce emails.
    Returns a set of email addresses that generated bounce-back notifications.
    """
    bounced_emails = set()
    try:
        print("\nChecking Gmail Inbox for bounce-back notifications...")
        mail = imaplib.IMAP4_SSL("imap.gmail.com")
        mail.login(SENDER_EMAIL, APP_PASSWORD)
        mail.select("INBOX")

        # Search for failure reports from mailer-daemon or subsystem
        status, messages = mail.search(
            None, '(FROM "mailer-daemon@googlemail.com" UNSEEN)'
        )
        if status != "OK" or not messages[0]:
            # Fallback search for subject containing 'Delivery Status Notification'
            status, messages = mail.search(
                None, '(SUBJECT "Delivery Status Notification (Failure)" UNSEEN)'
            )

        msg_ids = messages[0].split()
        print(f"Found {len(msg_ids)} bounce notification email(s).")

        for msg_id in msg_ids:
            res, msg_data = mail.fetch(msg_id, "(RFC822)")
            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email_parser.message_from_bytes(response_part[1])
                    body = ""
                    if msg.is_multipart():
                        for part in msg.walk():
                            if part.get_content_type() == "text/plain":
                                body += part.get_payload(decode=True).decode(
                                    "utf-8", errors="ignore"
                                )
                    else:
                        body = msg.get_payload(decode=True).decode(
                            "utf-8", errors="ignore"
                        )

                    # Extract failed recipient email from bounce body
                    emails_found = re.findall(
                        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", body
                    )
                    for em in emails_found:
                        if em.lower() != SENDER_EMAIL.lower():
                            bounced_emails.add(em.lower())

        mail.logout()
    except Exception as err:
        print(f"Warning: Could not complete IMAP bounce check: {err}")

    return bounced_emails


def send_bulk_emails_safely():
    excel_path = Path(EXCEL_FILE_PATH)
    if not excel_path.is_file():
        raise FileNotFoundError(f"Could not find Excel file at: {EXCEL_FILE_PATH}")

    df = pd.read_excel(excel_path)

    # Ensure tracking columns exist
    if "Last Sent Date" not in df.columns:
        df["Last Sent Date"] = None
    if "Status" not in df.columns:
        df["Status"] = None

    pdf_file = Path(ATTACHMENT_PATH)
    pdf_data = pdf_file.read_bytes() if pdf_file.is_file() else None
    pdf_name = pdf_file.name if pdf_file.is_file() else None

    now = datetime.now()
    cooldown_period = timedelta(days=7)

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(SENDER_EMAIL, APP_PASSWORD)
            print("Connected to Gmail SMTP.")

            total_rows = len(df)
            for idx, row in df.iterrows():
                name = str(row["Name"]).strip() if pd.notna(row.get("Name")) else ""
                email = str(row["Email"]).strip() if pd.notna(row.get("Email")) else ""

                if not name or not email or "@" not in email:
                    continue

                # 7-Day Cooldown Validation
                last_sent_val = row["Last Sent Date"]
                if pd.notna(last_sent_val):
                    try:
                        last_sent_dt = pd.to_datetime(last_sent_val)
                        if now - last_sent_dt < cooldown_period:
                            days_left = 7 - (now - last_sent_dt).days
                            print(
                                f"[{idx + 1}/{total_rows}] SKIPPED: {email} "
                                f"(Sent less than 7 days ago. Retry in ~{days_left} day(s))."
                            )
                            continue
                    except Exception:
                        pass

                # Attempt Email Dispatch
                contact = {"name": name, "email": email}
                msg = create_email(contact, pdf_data, pdf_name)
                timestamp_str = now.strftime("%Y-%m-%d %H:%M:%S")

                try:
                    server.send_message(msg)
                    df.at[idx, "Last Sent Date"] = timestamp_str
                    df.at[idx, "Status"] = "Sent"
                    df.to_excel(excel_path, index=False)

                    print(
                        f"[{idx + 1}/{total_rows}] Delivered to: {email} | "
                        f"Status: Sent"
                    )

                except smtplib.SMTPRecipientsRefused:
                    # Immediate bounce / invalid email address rejected by server
                    df.at[idx, "Last Sent Date"] = timestamp_str
                    df.at[idx, "Status"] = "Bounced"
                    df.to_excel(excel_path, index=False)
                    print(
                        f"[{idx + 1}/{total_rows}] FAILED: {email} | Status: Bounced (Recipient Refused)"
                    )

                except smtplib.SMTPException as smtp_err:
                    df.at[idx, "Status"] = f"Failed: {smtp_err}"
                    df.to_excel(excel_path, index=False)
                    print(f"[{idx + 1}/{total_rows}] FAILED: {email} | {smtp_err}")

                # Randomized 5 to 10-second delay
                wait_time = random.randint(5, 10)
                print(f"Waiting {wait_time}s before next send...")
                time.sleep(wait_time)

        print("\nAll eligible emails processed.")

    except Exception as e:
        print(f"SMTP Error: {e}")

    # Post-send asynchronous bounce verification via IMAP
    print("Waiting 15 seconds for mail server bounce-backs to settle...")
    time.sleep(15)

    bounced_addresses = check_bounces_via_imap()
    if bounced_addresses:
        print(f"Updating Excel with IMAP bounce results: {bounced_addresses}")
        df_updated = pd.read_excel(excel_path)
        for idx, row in df_updated.iterrows():
            current_email = str(row.get("Email", "")).strip().lower()
            if current_email in bounced_addresses:
                df_updated.at[idx, "Status"] = "Bounced"
        df_updated.to_excel(excel_path, index=False)
        print("Excel updated with IMAP bounce statuses.")


if __name__ == "__main__":
    send_bulk_emails_safely()

Connected to Gmail SMTP.
[1/2908] SKIPPED: anurag.solankey@ambujacement.com (Sent less than 7 days ago. Retry in ~7 day(s)).
[2/2908] SKIPPED: kayelldeemetaliks1@gmail.com (Sent less than 7 days ago. Retry in ~7 day(s)).
[3/2908] SKIPPED: madhavalloys12@gmail.com (Sent less than 7 days ago. Retry in ~7 day(s)).
[4/2908] SKIPPED: bassialloys1@gmail.com (Sent less than 7 days ago. Retry in ~7 day(s)).
[5/2908] SKIPPED: dilipkumar.bera@cairnindia.com (Sent less than 7 days ago. Retry in ~7 day(s)).
[6/2908] SKIPPED: rajamansuman82@gmail.com (Sent less than 7 days ago. Retry in ~7 day(s)).
[7/2908] SKIPPED: dharmesh.patel@gpcpl.net (Sent less than 7 days ago. Retry in ~7 day(s)).
[8/2908] SKIPPED: gopalareddy.a@ksk.co.in (Sent less than 7 days ago. Retry in ~7 day(s)).
[9/2908] SKIPPED: hamirn@starpipeproducts.com (Sent less than 7 days ago. Retry in ~7 day(s)).
[10/2908] SKIPPED: vamlenv@vedanta.co.in (Sent less than 7 days ago. Retry in ~7 day(s)).
[11/2908] SKIPPED: imsalmakhan197555@gm

KeyboardInterrupt: 